# Validación offline del modelo

Este notebook se ejecuta desde la carpeta del modelo y no usa Internet.

In [ ]:
import os

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

In [ ]:
from pathlib import Path
import json
import os
import sys
import importlib.metadata

def resolve_model_path():
    configured = os.environ.get('MODEL_PATH', '').strip()
    if configured:
        return Path(configured).expanduser().resolve()
    if os.environ.get('DATABRICKS_RUNTIME_VERSION'):
        try:
            dbutils.widgets.text('model_path', '')
            configured = dbutils.widgets.get('model_path').strip()
        except Exception:
            configured = ''
        if configured:
            return Path(configured).resolve()
        raise RuntimeError('En Databricks indique MODEL_PATH o el widget model_path con una ruta /Volumes/...')
    return Path.cwd().resolve()

MODEL_PATH = resolve_model_path()
runtime = os.environ.get('DATABRICKS_RUNTIME_VERSION', 'local')
print('PREFLIGHT')
print(f'Python: {sys.version.split()[0]}')
print(f'DATABRICKS_RUNTIME_VERSION: {runtime}')
if runtime != 'local' and '17.3' in runtime and '-cpu-ml-' not in runtime:
    raise RuntimeError('Runtime estándar detectado: use ML Runtime; no intente pip install sin salida a PyPI.')
for package in ('torch', 'transformers', 'sentence-transformers'):
    try:
        print(f'{package}: {importlib.metadata.version(package)}')
    except importlib.metadata.PackageNotFoundError as exc:
        raise RuntimeError(f'Falta {package}; use ML Runtime. No se puede instalar desde PyPI.') from exc
try:
    assert MODEL_PATH.is_dir(), f'No existe MODEL_PATH: {MODEL_PATH}'
    next(MODEL_PATH.iterdir(), None)
except PermissionError as exc:
    raise RuntimeError('Sin acceso al Volume. En SINGLE_USER se requieren USE CATALOG, USE SCHEMA y READ VOLUME para el principal del cluster.') from exc
LFS_PREFIX = b'version https://git-lfs.github.com/spec/v1'
for artifact in MODEL_PATH.rglob('*'):
    if artifact.is_file() and artifact.open('rb').read(len(LFS_PREFIX)).startswith(LFS_PREFIX):
        raise RuntimeError(f'Puntero Git LFS detectado: {artifact.name}. Ejecute git lfs pull.')
print('PREFLIGHT OK')
metadata_path = MODEL_PATH / 'model-metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
required_fields = {'name', 'model_type', 'source', 'revision', 'framework', 'python_target'}
missing_fields = required_fields - metadata.keys()
assert not missing_fields, f'Metadata incompleta: {sorted(missing_fields)}'
assert metadata['name'] == MODEL_PATH.name, 'El nombre no coincide con la carpeta'
assert metadata['model_type'] == 'embedding', 'Tipo de modelo incorrecto'
if sys.version_info[:2] != (3, 11):
    print('ADVERTENCIA: minor de Python distinto al staging; continúe solo si la inferencia real pasa.')
print(f'Model path: {MODEL_PATH.resolve()}')
print(f'Model path: {MODEL_PATH.resolve()}')
for field in ('name', 'model_type', 'source', 'revision', 'framework', 'python_target'):
    print(f'{field}: {metadata[field]}')

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=False,
)

In [ ]:
import numpy as np

texts = [
    'Cliente solicita financiamiento para capital de trabajo.',
    'La empresa presenta crecimiento sostenido de ventas.',
    'La compañía mantiene una posición financiera estable.',
]
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=False,
)
assert embeddings.shape[0] == len(texts)
assert embeddings.ndim == 2
assert embeddings.shape[1] > 0
assert np.isfinite(embeddings).all()
print(f'modelo: {metadata["name"]}')
print(f'cantidad de textos: {len(texts)}')
print(f'shape: {embeddings.shape}')
print(f'dimensión del embedding: {embeddings.shape[1]}')
print(f'dtype: {embeddings.dtype}')
print('resultado: OK')

In [ ]:
print('VALIDATION OK')
print('Modelo cargado y ejecutado usando únicamente archivos locales.')